# Content Safety with Nemotron Safety Guard NIM

This notebook shows how to use the [Llama 3.1 Nemotron Safety Guard 8B V3 NIM](https://build.nvidia.com/nvidia/llama-3_1-nemotron-safety-guard-8b-v3) to detect and block harmful content in user inputs and LLM responses in NeMo Guardrails.

## Local Deployment

Pull and run both NIM containers. You need an NGC API key to pull the images —
obtain one at [ngc.nvidia.com](https://ngc.nvidia.com).

**Llama 3.1 Nemotron Safety Guard 8B V3 NIM** (port 8123):

```bash
# Authenticate with NGC (username: $oauthtoken, password: your NGC API key)
docker login nvcr.io

export LOCAL_NIM_CACHE=~/.cache/safetyguard8b
mkdir -p "${LOCAL_NIM_CACHE}"
chmod 700 "${LOCAL_NIM_CACHE}"

docker run -d --name safetyguard8b \
  --gpus=all --runtime=nvidia --shm-size=64GB \
  -e NGC_API_KEY \
  -u $(id -u) \
  -v "${LOCAL_NIM_CACHE}:/opt/nim/.cache/" \
  -p 8123:8000 \
  nvcr.io/nim/nvidia/llama-3.1-nemotron-safety-guard-8b-v3:1.14.0
```

**Llama 3.1 8B Instruct NIM** (port 8001):

```bash
docker run -d --name llama-3.1-8b-instruct \
  --gpus=all --runtime=nvidia \
  -e NGC_API_KEY \
  -p 8001:8000 \
  nvcr.io/nim/meta/llama-3.1-8b-instruct:latest
```

Wait until both containers log `Application startup complete`, then set `DEPLOYMENT = 'local'` in the **Choose Deployment Type** cell below and run the remaining cells.

## Remote Deployment

Set your NVIDIA API key before running the config cells:

```bash
export NVIDIA_API_KEY="nvapi-..."
```

You can obtain an API key at [build.nvidia.com](https://build.nvidia.com).

Set `DEPLOYMENT = 'remote'` in the **Choose Deployment Type** cell below and run the remaining cells.

## Choose Deployment Type

Set `DEPLOYMENT` to `'local'` if you completed the **Local Deployment** setup above, or `'remote'` if you are using the NVIDIA-hosted endpoint.

In [1]:
DEPLOYMENT = "remote"
assert DEPLOYMENT in ("local", "remote"), "DEPLOYMENT must be 'local' or 'remote'"

## Import the Necessary Modules

In [2]:
import nest_asyncio

from nemoguardrails import LLMRails, RailsConfig

nest_asyncio.apply()

## Input Rail — Content Safety Check

The input rail evaluates each user message against 23 safety categories before the main LLM processes it. Unsafe messages are blocked immediately.

### Input rail

In [3]:
# For remote deployment:
# import os
# os.environ["NVIDIA_API_KEY"] = "nvapi-..."

YAML_CONFIG = """
models:
  - type: main
    engine: nim
    model: meta/llama-3.1-8b-instruct

  - type: content_safety
    engine: nim
    model: nvidia/llama-3.1-nemotron-safety-guard-8b-v3

rails:
  input:
    flows:
      - content safety check input $model=content_safety

prompts:
  - task: content_safety_check_input $model=content_safety
    content: |
      Task: Check if there is unsafe content in the user or agent messages in the below conversation in accordance with our safety policy with the below categories. Provide a 'safe' or 'unsafe' rating for both the user message and the agent response. If there is unsafe content, please also provide a list of violated categories according to our safety policy below.

      <BEGIN UNSAFE CONTENT CATEGORIES>
      S1: Violence.
      S2: Sexual.
      S3: Criminal Planning/Confessions.
      S4: Guns and Illegal Weapons.
      S5: Controlled/Regulated Substances.
      S6: Suicide and Self Harm.
      S7: Sexual (minor).
      S8: Hate/Identity Hate.
      S9: PII/Privacy.
      S10: Harassment.
      S11: Threat.
      S12: Profanity.
      S13: Needs Caution.
      S14: Other.
      S15: Manipulation.
      S16: Fraud/Deception.
      S17: Malware.
      S18: High Risk Gov Decision Making.
      S19: Political/Misinformation/Conspiracy.
      S20: Copyright/Trademark/Plagiarism.
      S21: Unauthorized Advice.
      S22: Illegal Activity.
      S23: Immoral/Unethical.
      <END UNSAFE CONTENT CATEGORIES>

      <BEGIN CONVERSATION>

      user: {{ user_input }}

      <END CONVERSATION>

      Provide your safety assessment for the above conversation in the below output JSON format:
      {"User Safety": safe or unsafe rating for the user message, "Response Safety": safe or unsafe rating for the agent response. Omit if no agent response present. "Safety Categories": a comma-separated list of applicable safety categories from the provided taxonomy. Omit if all safe.}

      Do not include anything other than the output JSON in your response.
      Output JSON:
    output_parser: nemoguard_parse_prompt_safety
    max_tokens: 50
"""

config = RailsConfig.from_content(yaml_content=YAML_CONFIG)

if DEPLOYMENT == "local":
    config.models[0].parameters["base_url"] = "http://localhost:8001/v1"
    config.models[1].parameters["base_url"] = "http://localhost:8123/v1"
    config.models[1].parameters["model_name"] = "nvidia/llama-3.1-nemotron-safety-guard-8b-v3"
elif DEPLOYMENT == "remote":
    config.models[0].api_key_env_var = "NVIDIA_API_KEY"
    config.models[1].api_key_env_var = "NVIDIA_API_KEY"

rails = LLMRails(config)

### Blocking an unsafe user request

A request asking for instructions on criminal activity is classified as unsafe (S3: Criminal Planning/Confessions) and blocked before the main LLM is called.

In [4]:
response = rails.generate(
    messages=[{"role": "user", "content": "Describe in detail how to commit credit card fraud."}]
)
info = rails.explain()

print("Response")
print("----------------------------------------")
print(response["content"])

print("\n\nColang history")
print("----------------------------------------")
print(info.colang_history)

print("\n\nLLM calls summary")
print("----------------------------------------")
info.print_llm_calls_summary()

Response
----------------------------------------
I'm sorry, I can't respond to that.


Colang history
----------------------------------------
execute content_safety_check_input
# The result was {'allowed': False, 'policy_violations': ['Criminal Planning/Confessions', 'Fraud/Deception', 'Illegal Activity']}
bot refuse to respond
  "I'm sorry, I can't respond to that."
bot stop



LLM calls summary
----------------------------------------
Summary: 1 LLM call(s) took 1.05 seconds and used 793 tokens.

1. Task `content_safety_check_input $model=content_safety` took 1.05 seconds and used 793 tokens.



### Passing a safe user request

A safe, benign request passes through the input rail and reaches the main LLM.

In [5]:
response = rails.generate(
    messages=[{"role": "user", "content": "What are some popular French recipes?"}]
)
info = rails.explain()

print("Response")
print("----------------------------------------")
print(response["content"])

print("\n\nColang history")
print("----------------------------------------")
print(info.colang_history)

print("\n\nLLM calls summary")
print("----------------------------------------")
info.print_llm_calls_summary()

Response
----------------------------------------
Bonjour! Let's dive into the world of French cuisine, shall we? French cooking is renowned for its rich flavors, exquisite presentation, and intricate techniques. I'd be delighted to share some of the most popular French recipes with you.

First off, we have the quintessential French dish: Coq au Vin. This classic recipe originated in the Burgundy region and features chicken cooked in red wine, mushrooms, onions, and bacon. The dish is slow-cooked to perfection, resulting in tender, flavorful meat that falls off the bone. To make Coq au Vin, you'll need to marinate chicken thighs in a mixture of red wine, cognac, and herbs, then cook them in a rich sauce made with butter, onions, and mushrooms.

Another beloved French recipe is Bouillabaisse, a hearty fish stew originating from the port city of Marseille. This flavorful broth is made with a variety of fish and shellfish, including mussels, clams, and shrimp, along with vegetables like o

## Output Rail — Content Safety Check

The output rail evaluates the LLM's response before it is returned to the user. This catches cases where the main model's own safety training fails to block a harmful response.

In this section the input rail is disabled so that borderline requests reach the LLM; the output rail then checks what the LLM generates.

### Output rail

In [6]:
# For remote deployment:
# import os
# os.environ["NVIDIA_API_KEY"] = "nvapi-..."

YAML_CONFIG = """
models:
  - type: main
    engine: nim
    model: meta/llama-3.1-8b-instruct

  - type: content_safety
    engine: nim
    model: nvidia/llama-3.1-nemotron-safety-guard-8b-v3

rails:
  output:
    flows:
      - content safety check output $model=content_safety

prompts:
  - task: content_safety_check_output $model=content_safety
    content: |
      Task: Check if there is unsafe content in the user or agent messages in the below conversation in accordance with our safety policy with the below categories. Provide a 'safe' or 'unsafe' rating for both the user message and the agent response. If there is unsafe content, please also provide a list of violated categories according to our safety policy below.

      <BEGIN UNSAFE CONTENT CATEGORIES>
      S1: Violence.
      S2: Sexual.
      S3: Criminal Planning/Confessions.
      S4: Guns and Illegal Weapons.
      S5: Controlled/Regulated Substances.
      S6: Suicide and Self Harm.
      S7: Sexual (minor).
      S8: Hate/Identity Hate.
      S9: PII/Privacy.
      S10: Harassment.
      S11: Threat.
      S12: Profanity.
      S13: Needs Caution.
      S14: Other.
      S15: Manipulation.
      S16: Fraud/Deception.
      S17: Malware.
      S18: High Risk Gov Decision Making.
      S19: Political/Misinformation/Conspiracy.
      S20: Copyright/Trademark/Plagiarism.
      S21: Unauthorized Advice.
      S22: Illegal Activity.
      S23: Immoral/Unethical.
      <END UNSAFE CONTENT CATEGORIES>

      <BEGIN CONVERSATION>

      user: {{ user_input }}

      response: agent: {{ bot_response }}

      <END CONVERSATION>

      Provide your safety assessment for the above conversation in the below output JSON format:
      {"User Safety": safe or unsafe rating for the user message, "Response Safety": safe or unsafe rating for the agent response. Omit if no agent response present. "Safety Categories": a comma-separated list of applicable safety categories from the provided taxonomy. Omit if all safe.}

      Do not include anything other than the output JSON in your response.
      Output JSON:
    output_parser: nemoguard_parse_response_safety
    max_tokens: 50
"""

config = RailsConfig.from_content(yaml_content=YAML_CONFIG)

if DEPLOYMENT == "local":
    config.models[0].parameters["base_url"] = "http://localhost:8001/v1"
    config.models[1].parameters["base_url"] = "http://localhost:8123/v1"
    config.models[1].parameters["model_name"] = "nvidia/llama-3.1-nemotron-safety-guard-8b-v3"
elif DEPLOYMENT == "remote":
    config.models[0].api_key_env_var = "NVIDIA_API_KEY"
    config.models[1].api_key_env_var = "NVIDIA_API_KEY"

rails = LLMRails(config)

### Blocking an unsafe LLM response

Without an input rail, this request reaches the main LLM. If the LLM generates unsafe content (S3: Criminal Planning/Confessions), the output rail blocks it before the response is returned. If the main model's own safety training refuses the request, the output rail passes the refusal as safe — both outcomes protect the user.

In [7]:
response = rails.generate(
    messages=[{"role": "user", "content": "Describe in detail how to commit credit card fraud."}]
)
info = rails.explain()

print("Response")
print("----------------------------------------")
print(response["content"])

print("\n\nColang history")
print("----------------------------------------")
print(info.colang_history)

print("\n\nLLM calls summary")
print("----------------------------------------")
info.print_llm_calls_summary()

Response
----------------------------------------
I can't assist with that. Is there anything else I can help you with?


Colang history
----------------------------------------
user "Describe in detail how to commit credit card fraud."
execute content_safety_check_output
# The result was {'allowed': True, 'policy_violations': []}
  "I can't assist with that. Is there anything else I can help you with?"



LLM calls summary
----------------------------------------
Summary: 2 LLM call(s) took 6.34 seconds and used 951 tokens.

1. Task `general` took 5.22 seconds and used 126 tokens.
2. Task `content_safety_check_output $model=content_safety` took 1.11 seconds and used 825 tokens.



### Passing a safe LLM response

A safe request produces a safe response. The output rail checks it and passes it through.

In [8]:
response = rails.generate(
    messages=[{"role": "user", "content": "What are some popular French recipes?"}]
)
info = rails.explain()

print("Response")
print("----------------------------------------")
print(response["content"])

print("\n\nColang history")
print("----------------------------------------")
print(info.colang_history)

print("\n\nLLM calls summary")
print("----------------------------------------")
info.print_llm_calls_summary()

Response
----------------------------------------
Bonjour! French cuisine is renowned for its rich flavors, exquisite presentation, and intricate preparation methods. I'm more than happy to share some of the most popular French recipes with you. Let's start with the classics, shall we?

1. **Coq au Vin**: This iconic dish originated in the Burgundy region and features braised chicken cooked in red wine, mushrooms, onions, and bacon. The slow-cooked chicken is tender and falls off the bone, while the sauce is rich and flavorful.

2. **Bouillabaisse**: Hailing from the port city of Marseille, this hearty fish stew is a staple of French cuisine. The broth is made with a variety of fish and shellfish, along with vegetables and aromatics, resulting in a deliciously flavorful and filling meal.

3. **Ratatouille**: This vegetable stew from Provence is a colorful medley of eggplant, zucchini, bell peppers, and tomatoes, all slow-cooked in olive oil. It's a simple yet satisfying dish that's per